# Lesson 20: Self-Critique Loop — LLM Checks Its Own Arithmetic

## Objective
Build a **reflection loop** where a Solver LLM answers the math question, then a Critic LLM checks the answer for correctness, and the loop retries until the answer is verified correct.

## Limitation of Lesson 19
- ❌ LLM answers were taken at face value — no verification
- ❌ LLMs can make arithmetic errors, especially with large numbers
- ❌ No self-correction mechanism

## What is NEW in Lesson 20?
- ✅ **Solver → Critic → [loop?]** reflection pattern
- ✅ Two-LLM architecture: Solver + independent Critic
- ✅ Cycle with attempt counter (Lesson 5 pattern applied to LLMs)
- ✅ Structured critique: the Critic outputs CORRECT/INCORRECT + correction
- ✅ **Self-healing agent** that corrects its own errors

## Theory: Reflection Loop

```
START → solve → critique → [correct?]
                               ├── YES → END  (verified answer)
                               └── NO  → revise → critique  (loop)
```

### Why use two separate LLM calls?
- The **Solver** is optimistic: focuses on computing the answer
- The **Critic** is skeptical: focuses on finding errors
- Using the SAME LLM for both often results in it agreeing with itself
- A separate Critic call (or even a different model) gives more honest feedback

### Production applications
- Code generation + code review agent
- SQL generation + SQL validation agent
- Medical reasoning + safety checker
- Any domain where errors are costly


In [ ]:
# ── Cell 1: Imports + LLM Setup ──────────────────────────────────────────────
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

import os
import vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Optional, Literal, Annotated
import operator
from IPython.display import Image, display

load_dotenv()
vertexai.init(project=os.getenv("PROJECT_ID"), location=os.getenv("LOCATION"))

# Both Solver and Critic use the same LLM but with DIFFERENT prompts
# In production: consider using different models or temperatures
llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)

MAX_RETRIES = 3
print(f"✓ Setup complete | MAX_RETRIES = {MAX_RETRIES}")


In [ ]:
# ── Cell 2: State Schema ────────────────────────────────────────────────────
class State(TypedDict):
    question: str
    proposed_answer: str          # Solver's current answer
    proposed_result: Optional[float]  # Numeric result from solver
    critique: str                 # Critic's feedback
    is_correct: bool              # Did critic approve?
    correct_result: Optional[float]   # Critic's computed correct answer
    retries: int                  # How many times we've revised
    history: Annotated[list, operator.add]
    final_answer: str             # Verified answer

print("✓ State defined")


In [ ]:
# ── Cell 3: Solver Node ──────────────────────────────────────────────────────
SOLVER_PROMPT = """You are an arithmetic solver. Compute the answer to the math question.
Show your work step by step.
End your response with: ANSWER: <number>
Be precise with calculations."""

def solver_node(state: State) -> dict:
    retries = state.get("retries", 0)
    
    # If we have a previous critique, include it to guide the revision
    messages = [SystemMessage(content=SOLVER_PROMPT)]
    if state.get("critique") and retries > 0:
        messages.append(SystemMessage(
            content=f"Your previous answer was WRONG. Critic feedback: {state['critique']}\n"
                   f"Please correct your answer."
        ))
    messages.append(HumanMessage(content=state["question"]))
    
    print(f"  [solver] Attempt {retries+1}: solving '{state['question']}'")
    resp = llm.invoke(messages)
    content = resp.content.strip()
    print(f"  [solver] Proposed: {content[:80]}")
    
    # Extract ANSWER: number
    result = None
    for line in content.split("\n"):
        if "ANSWER:" in line:
            try:
                result = float(line.split("ANSWER:")[1].strip())
            except:
                pass
    
    entry = f"Solver (attempt {retries+1}): {content[:60]}..."
    return {
        "proposed_answer": content,
        "proposed_result": result,
        "history": [entry]
    }

print("✓ Solver node defined")


In [ ]:
# ── Cell 4: Critic Node ──────────────────────────────────────────────────────
CRITIC_PROMPT = """You are an arithmetic CRITIC. Your job is to verify if the given answer is CORRECT.
Independently compute the answer yourself and compare.

Respond in EXACTLY this format:
VERDICT: CORRECT or INCORRECT
CORRECT_ANSWER: <number>
REASON: <brief explanation>

Be strict — if the number is even slightly wrong, say INCORRECT."""

def critic_node(state: State) -> dict:
    question = state["question"]
    proposed = state["proposed_answer"]
    
    messages = [
        SystemMessage(content=CRITIC_PROMPT),
        HumanMessage(content=f"Question: {question}\n\nProposed Answer:\n{proposed}")
    ]
    
    print(f"  [critic]  Reviewing: '{proposed[:60]}...'")
    resp = llm.invoke(messages)
    content = resp.content.strip()
    print(f"  [critic]  Verdict: {content[:80]}")
    
    # Parse verdict
    lines = {}
    for line in content.split("\n"):
        if ":" in line:
            k, _, v = line.partition(":")
            lines[k.strip().upper()] = v.strip()
    
    verdict    = lines.get("VERDICT", "INCORRECT").upper()
    is_correct = (verdict == "CORRECT")
    reason     = lines.get("REASON", "")
    
    try:
        correct_result = float(lines.get("CORRECT_ANSWER", "0"))
    except:
        correct_result = state.get("proposed_result")
    
    entry = f"Critic: {'✓ CORRECT' if is_correct else '✗ INCORRECT'} | {reason[:50]}"
    return {
        "critique": f"{verdict}: {reason}",
        "is_correct": is_correct,
        "correct_result": correct_result,
        "retries": state.get("retries", 0) + 1,
        "history": [entry]
    }

print("✓ Critic node defined")


In [ ]:
# ── Cell 5: Finalize Node ────────────────────────────────────────────────────
def finalize_node(state: State) -> dict:
    result = state["correct_result"] if state["is_correct"] else state["correct_result"]
    answer = f"Verified answer: {result}"
    print(f"  [finalize] {answer}")
    return {"final_answer": answer}

print("✓ Finalize node defined")


In [ ]:
# ── Cell 6: Critique Router ──────────────────────────────────────────────────
def critique_router(state: State) -> Literal["finalize", "solver", "give_up"]:
    if state["is_correct"]:
        print(f"  [router] ✓ Answer verified — finalizing")
        return "finalize"
    elif state["retries"] >= MAX_RETRIES:
        print(f"  [router] ✗ Max retries ({MAX_RETRIES}) — giving up")
        return "give_up"
    else:
        print(f"  [router] ✗ Incorrect — retry {state['retries']}/{MAX_RETRIES}")
        return "solver"

def give_up_node(state: State) -> dict:
    answer = f"Could not verify after {MAX_RETRIES} attempts. Best guess: {state['proposed_result']}"
    print(f"  [give_up] {answer}")
    return {"final_answer": answer}

print("✓ Critique router and give_up node defined")


In [ ]:
# ── Cell 7: Build Self-Critique Graph ────────────────────────────────────────
builder = StateGraph(State)

builder.add_node("solver",   solver_node)
builder.add_node("critic",   critic_node)
builder.add_node("finalize", finalize_node)
builder.add_node("give_up",  give_up_node)

# Flow
builder.add_edge(START,    "solver")   # Start by solving
builder.add_edge("solver", "critic")   # Always critique after solving

# After critique: finalize (correct), retry (wrong), or give up (max retries)
builder.add_conditional_edges("critic", critique_router, {
    "finalize": "finalize",
    "solver":   "solver",    # ← LOOP back to solver if wrong
    "give_up":  "give_up",
})

builder.add_edge("finalize", END)
builder.add_edge("give_up",  END)

graph = builder.compile()
print("✓ Self-critique graph compiled")
print("  Loop: solver → critic → [wrong?] → solver (retry)")


In [ ]:
# ── Cell 8: Visualize ───────────────────────────────────────────────────────
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# ── Cell 9: Test with Straightforward Questions ──────────────────────────────
def run(question: str):
    state = {
        "question": question,
        "proposed_answer": "", "proposed_result": None,
        "critique": "", "is_correct": False, "correct_result": None,
        "retries": 0, "history": [], "final_answer": ""
    }
    print(f"\nQuestion: '{question}'")
    print("-" * 55)
    out = graph.invoke(state)
    print("-" * 55)
    print(f"Final: {out['final_answer']}")
    print(f"Attempts: {out['retries']} | Verified: {out['is_correct']}")
    return out

run("what is 23 multiplied by 17?")


In [ ]:
run("divide 1764 by 42")


In [ ]:
run("what is 13 plus 29 plus 47?")


In [ ]:
# ── Cell 10: Show Full History ───────────────────────────────────────────────
out = run("what is 256 divided by 16?")
print("\nFull history:")
for i, entry in enumerate(out["history"], 1):
    print(f"  {i}. {entry[:80]}")


## Reflection Loop Architecture

```
START
  ↓
solver (LLM generates answer)
  ↓
critic (LLM independently verifies)
  ↓
router ─── CORRECT ──→ finalize → END
       └── WRONG   ──→ solver   ← LOOP (includes critique feedback)
       └── MAX     ──→ give_up → END
```

### Key Design Decisions

| Decision | Why |
|---------|-----|
| Separate Critic | Independent verification (not just "agree with yourself") |
| Include critique in solver retry | Solver gets specific feedback on what was wrong |
| Attempt counter | Prevents infinite loop |
| CORRECT_ANSWER in critique | Critic provides the fix, not just the verdict |

### Production Enhancement Ideas
```python
# Use different temperature for solver vs critic
solver_llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0.3)  # Some creativity
critic_llm  = ChatVertexAI(model="gemini-2.5-flash", temperature=0)    # Strict/deterministic

# Use a stronger model for critic
critic_llm = ChatVertexAI(model="gemini-2.5-pro", temperature=0)
```

## Summary

| | Lesson 12 (Supervisor) | Lesson 20 (Self-Critique) |
|--|----------------------|--------------------------|
| Verification | None | Automatic |
| Correction | None | LLM self-corrects |
| Loop | No | Yes (until verified) |

## What Lesson 21 Shows
- ✅ All 20 lessons combined into a production-grade architecture
- ✅ Complete arithmetic agent with all LangGraph features integrated
